# ETL SSC — Avance 03

# Proyecto A&C — Avance 03
**Análisis de accidentes y hechos de tránsito en la Ciudad de México**  
**Equipo:** Manuel Soto Romero y Araceli Liliana Reyes Cabello · UACM · Inteligencia de Negocios · Grupo 313 · 2026-2

## Objetivo del notebook
En este notebook explico y ejecuto el proceso correspondiente a esta fuente. Trabajo sobre copias, conservo las columnas originales y genero archivos independientes para los registros conservados, duplicados, excluidos y pendientes. Cada decisión queda asociada con el código de regla y su justificación.

**Pregunta de negocio:** ¿En qué alcaldías y franjas horarias de la Ciudad de México se registra la mayor cantidad de personas lesionadas en hechos de tránsito, de acuerdo con los registros disponibles a partir de 2018?

**Importante:** los CSV originales deben colocarse en `data/raw/`. Los resultados numéricos solo se producen al ejecutar con esos archivos; este notebook no inventa cifras ni considera hallazgos preliminares del Avance 02 como resultados nuevos.

## Alcance de esta fuente

La SSC contiene cinco tablas con unidades distintas: hechos, personas por edad, personas por sexo, tipo de persona y vehículos. Se procesan por separado y no se unen en esta etapa.

**Reglas aplicadas:** RG-01 a RG-12; SSC-R01 a SSC-R07. La validación de edad SSC-R04 se ejecuta solo en archivos cuya columna de edad se identifique. Las cinco tablas no se mezclan.

## 1. Entorno, rutas y bitácora

Creo una carpeta por ejecución para no sobrescribir resultados anteriores. Los originales permanecen en `data/raw` y nunca se exportan encima.

In [ ]:
# 1. IMPORTAR LIBRERÍAS
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

import pandas as pd
import numpy as np

import hashlib
import json
import re
import csv
import unicodedata

# 2. CONFIGURACIÓN GENERAL

TZ = ZoneInfo("America/Mexico_City")

# Directorio donde se encuentra el notebook
ROOT = Path.cwd()

print("Directorio actual:")
print(ROOT)

# 3. LOCALIZAR AUTOMÁTICAMENTE EL PROYECTO

PROYECTO = None
SSC_ETAPA3 = None

# Buscar la carpeta del proyecto en el directorio actual y en todos sus directorios superiores.

for carpeta in [ROOT, *ROOT.parents]:

    posible_ruta = (
        carpeta
        / "resultados"
        / "resultados_SSC"
        / "etapa3"
    )

    if posible_ruta.is_dir():
        PROYECTO = carpeta
        SSC_ETAPA3 = posible_ruta
        break

# Comprobar que se encontró la carpeta
if SSC_ETAPA3 is None:

    raise FileNotFoundError(
        "No se encontró la carpeta "
        "'resultados/resultados_SSC/etapa3'. "
        "Verifica la ubicación de los archivos."
    )

# La carpeta etapa3 será la entrada de datos
RAW = SSC_ETAPA3

print("\nPROYECTO LOCALIZADO:")
print(PROYECTO)

print("\nCARPETA DE ENTRADA SSC:")
print(RAW)

# 4. CREAR IDENTIFICADOR DE EJECUCIÓN

RUN_ID = datetime.now(TZ).strftime("%Y%m%d_%H%M%S")

print("\nIdentificador de ejecución:")
print(RUN_ID)

# 5. CREAR CARPETAS DE SALIDA DEL AVANCE 03

# Carpeta para los archivos procesados
OUT = (
    ROOT
    / "data"
    / "processed"
    / "SSC"
    / RUN_ID
)

# Carpeta para la bitácora automática
LOG_DIR = (
    ROOT
    / "bitacora"
    / "SSC"
    / RUN_ID
)

# Crear carpetas si no existen
OUT.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("\nCARPETAS CREADAS:")
print("Datos procesados:", OUT)
print("Bitácora:", LOG_DIR)

# 6. INICIALIZAR BITÁCORA

BITACORA = []

def loggear(
    archivo,
    paso,
    regla,
    antes,
    despues,
    decision,
    justificacion,
    detalle=""
):

    registro = {
        "fecha_hora_local": datetime.now(TZ).isoformat(
            timespec="seconds"
        ),

        "fuente": "SSC",
        "archivo_origen": str(archivo),
        "proceso": paso,
        "regla": regla,
        "registros_antes": int(antes),
        "registros_despues": int(despues),
        "decision": decision,
        "justificacion": justificacion,
        "detalle": detalle,
        "version_proceso": "Avance03-v2"
    }
    BITACORA.append(registro)

# 7. FUNCIÓN PARA CALCULAR SHA256

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as archivo:
        for bloque in iter(
            lambda: archivo.read(1024 * 1024),
            b""
        ):
            h.update(bloque)
    return h.hexdigest()

# 8. FUNCIÓN PARA EXPORTAR ARCHIVOS

def escribir(df, path):
    df.to_csv(
        path,
        index=False,
        encoding="utf-8-sig"
    )

    # Volver a leer para comprobar que se guardó correctamente
    reread = pd.read_csv(
        path,
        dtype="string",
        encoding="utf-8-sig"
    )
    assert len(reread) == len(df), (
        f"El conteo cambió al exportar {path}"
    )
    return path

# 9. NORMALIZAR NOMBRES DE COLUMNAS

def norm(s):
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        unicodedata.normalize(
            "NFKD",
            str(s)
        ).encode("ascii", "ignore").decode().lower()
    ).strip("_")

# 10. BUSCAR COLUMNAS POR SUS ALIAS

def buscar_columna(df, aliases):
    m = {
        norm(c): c
        for c in df.columns
    }
    for alias in aliases:
        if norm(alias) in m:
            return m[norm(alias)]
    return None

# 11. LEER CSV DE FORMA SEGURA

def leer_csv_seguro(path):

    # Se mantienen todos los datos como texto para evitar modificar folios, identificadores y ceros iniciales.

    return pd.read_csv(
        path,
        dtype="string",
        encoding="utf-8-sig",
        low_memory=False
    )

# 12. IDENTIFICAR LOS CINCO ARCHIVOS PRINCIPALES DE SSC

NOMBRES_PRINCIPALES = [

    "HechosTransito_SSC_etapa3.csv",
    "VehiculosInvolucrados_SSC_etapa3.csv",
    "PersonasLyFporSexo_SSC_etapa3.csv",
    "PersonasLyFporEdad_SSC_etapa3.csv",
    "PersonasLyFporTipoPersona_SSC_etapa3.csv"
]

# Buscar los archivos existentes
ARCHIVOS = [
    RAW / nombre
    for nombre in NOMBRES_PRINCIPALES
    if (RAW / nombre).is_file()
]

# Mostrar archivos encontrados
print("\n========================================")
print("ARCHIVOS SSC ENCONTRADOS")
print("========================================")

for archivo in ARCHIVOS:
    print("✓", archivo.name)
print("\nTotal de archivos encontrados:", len(ARCHIVOS))

# Comprobar que estén los cinco archivos
if len(ARCHIVOS) != 5:

    faltantes = [
        nombre
        for nombre in NOMBRES_PRINCIPALES
        if not (RAW / nombre).is_file()
    ]

    raise FileNotFoundError("Faltan archivos de SSC: "
        + ", ".join(faltantes)
    )

else:
    print("\nCorrecto: los cinco archivos están disponibles.")

# 13. CARGAR LOS CINCO ARCHIVOS EN DATAFRAMES

datos_ssc = {}

for archivo in ARCHIVOS:

    nombre = archivo.stem
    df = leer_csv_seguro(archivo)
    datos_ssc[nombre] = df

    print("\n----------------------------------------")
    print("ARCHIVO:", archivo.name)
    print("Filas:", df.shape[0])
    print("Columnas:", df.shape[1])
    print("----------------------------------------")

    display(df.head())
    
# 14. CREAR VARIABLES INDIVIDUALES

df_hechos = datos_ssc[
    "HechosTransito_SSC_etapa3"
]
df_vehiculos = datos_ssc[
    "VehiculosInvolucrados_SSC_etapa3"
]
df_personas_sexo = datos_ssc[
    "PersonasLyFporSexo_SSC_etapa3"
]
df_personas_edad = datos_ssc[
    "PersonasLyFporEdad_SSC_etapa3"
]
df_personas_tipo = datos_ssc[
    "PersonasLyFporTipoPersona_SSC_etapa3"
]

# 15. RESUMEN GENERAL DE LOS DATASETS

resumen_datasets = []

for nombre, df in datos_ssc.items():

    resumen_datasets.append({
        "archivo": nombre,
        "filas": df.shape[0],
        "columnas": df.shape[1],
        "valores_nulos": int(
            df.isna().sum().sum()
        ),
        "duplicados": int(
            df.duplicated().sum()
        )
    })

df_resumen = pd.DataFrame(resumen_datasets)

print("\n========================================")
print("RESUMEN GENERAL DE LOS DATASETS SSC")
print("========================================")

display(df_resumen)

# 16. REGISTRAR LA CARGA EN LA BITÁCORA

for archivo in ARCHIVOS:

    df = datos_ssc[archivo.stem]

    loggear(

        archivo=archivo.name,
        paso="Carga de datos",
        regla="Importación de archivo CSV",
        antes=len(df),
        despues=len(df),
        decision="Aceptado",
        justificacion=(
            "El archivo se cargó conservando los valores "
            "como texto para proteger identificadores."
        ),

        detalle=(
            f"Columnas: {df.shape[1]}; "
            f"SHA256: {sha256(archivo)}"
        )

    )

# 17. GUARDAR RESUMEN DE LOS DATASETS

escribir(
    df_resumen,
    OUT / "resumen_inicial_SSC.csv"
)

# 18. GUARDAR BITÁCORA INICIAL

df_bitacora = pd.DataFrame(BITACORA)

escribir(
    df_bitacora,
    LOG_DIR / "bitacora_SSC.csv"
)

# 19. RESULTADO FINAL

print("\n========================================")
print("CONFIGURACIÓN SSC COMPLETADA")
print("========================================")

print("Archivos cargados:", len(datos_ssc))
print("Filas de hechos de tránsito:", len(df_hechos))
print("Filas de vehículos:", len(df_vehiculos))
print("Filas de personas por sexo:", len(df_personas_sexo))
print("Filas de personas por edad:", len(df_personas_edad))
print("Filas de personas por tipo:", len(df_personas_tipo))

print("\nCarpeta de salida:")
print(OUT)

print("\nCarpeta de bitácora:")
print(LOG_DIR)

print("\nBitácora inicial guardada correctamente.")


## 2. Extracción e inventario

Leo cada CSV de la fuente como texto, registro dimensiones, columnas y SHA-256. Mantengo una tabla por archivo porque pueden tener unidades de observación diferentes.

In [ ]:

inventario=[]
tablas={}
for p in ARCHIVOS:
    d=leer_csv_seguro(p)
    tablas[p.name]=d.copy(deep=True)
    inventario.append({"archivo":p.name,"ruta":str(p),"filas":len(d),"columnas":len(d.columns),
                       "sha256":sha256(p),"columnas_detectadas":" | ".join(map(str,d.columns))})
    loggear(p.name,"Extracción","RG-01/RG-11",len(d),len(d),"Conservar copia de trabajo",
            "El original no se modifica; se registra procedencia y huella SHA-256.")
inventario_df=pd.DataFrame(inventario)
display(inventario_df)
escribir(inventario_df,OUT/"inventario_fuentes.csv")


## 3. Duplicados exactos y folios repetidos

La comparación de duplicados exactos usa todas las columnas originales. Se conserva la primera aparición y se exportan las copias adicionales a un CSV independiente. Compartir folio no basta para eliminar una fila: se crea un reporte de folios repetidos para revisión.

In [ ]:

# 20. PREPARAR DATASETS PARA EL PROCESAMIENTO ETL

resultados = {}

for nombre, df in datos_ssc.items():
    resultados[nombre] = {"base": df.copy()}
    print(f"Dataset preparado: {nombre} | Filas: {len(df)}")


# 21. VALIDACIÓN DE EDADES Y CLASIFICACIÓN DE REGISTROS SSC

RESUMEN_EDADES = []

for nombre, r in resultados.items():

    base = r["base"].copy()
    base["_decision"] = "conservado"
    base["_motivo_decision"] = "Cumple las reglas detectables automáticamente"
    base["_regla_aplicada"] = "RG-10"
    base["_valores_ausentes"] = base.isna().sum(axis=1)

    # Identificar valores SD y campos vacíos
    for col in list(base.columns):
        if pd.api.types.is_string_dtype(base[col]):
            valores = base[col].astype("string").fillna("").str.strip()
            base[f"_{norm(col)}_es_SD"] = valores.str.upper().isin(["SD", "S/D", "NO DISPONIBLE"])
            base[f"_{norm(col)}_es_vacio"] = valores.eq("")

    # Localizar columna de edad
    col_edad = buscar_columna(base, ["edad", "edad_persona", "años", "anios"])

    if col_edad is not None:

        # Convertir edad a formato numérico sin modificar el dato original
        edad_texto = base[col_edad].astype("string").fillna("").str.strip()
        edad_num = pd.to_numeric(edad_texto.str.replace(",", ".", regex=False), errors="coerce")

        # Identificar edades válidas entre 0 y 99
        edades_validas = edad_num[edad_num.notna() & edad_num.ge(0) & edad_num.le(99)]
        edad_min = float(edades_validas.min()) if len(edades_validas) else None
        edad_min_ref = max(0, edad_min - 1) if edad_min is not None else 0

        base["_edad_numerica"] = edad_num
        base["_edad_minima_referencia"] = edad_min_ref
        base["_edad_validacion"] = "valida"

        # Detectar errores de edad
        mask_no_num = edad_num.isna() & edad_texto.ne("")
        mask_vacia = edad_texto.eq("")
        mask_neg = edad_num.lt(0)
        mask_mayor = edad_num.gt(99)

        base.loc[mask_no_num, "_edad_validacion"] = "no numerica"
        base.loc[mask_vacia, "_edad_validacion"] = "ausente"
        base.loc[mask_neg, "_edad_validacion"] = "negativa"
        base.loc[mask_mayor, "_edad_validacion"] = "mayor a 99"

        # Clasificar registros según las reglas
        decisiones = [
            (mask_no_num, "pendiente", "Edad no numérica; requiere revisión", "SSC-R04 / RG-10"),
            (mask_vacia, "pendiente", "Edad ausente; no se imputa", "RG-03 / SSC-R04"),
            (mask_neg, "pendiente", "Edad negativa fuera de rango", "SSC-R04"),
            (mask_mayor, "excluido", "Edad superior a 99; excluida del análisis de edad", "SSC-R04")
        ]

        for mask, decision, motivo, regla in decisiones:
            base.loc[mask, "_decision"] = decision
            base.loc[mask, "_motivo_decision"] = motivo
            base.loc[mask, "_regla_aplicada"] = regla

        # Conservar registros que cumplen la validación
        mask_valida = ~(mask_no_num | mask_vacia | mask_neg | mask_mayor)
        base.loc[mask_valida, "_decision"] = "conservado"
        base.loc[mask_valida, "_motivo_decision"] = "Cumple regla de edad"
        base.loc[mask_valida, "_regla_aplicada"] = "SSC-R04"

        # Guardar edades mayores a 99 para revisión
        mayores = base.loc[mask_mayor].copy()
        mayores["_destino"] = "No usar en cálculos de edad; revisar valor original"
        escribir(mayores, OUT / f"{Path(str(nombre)).stem}_edad_mayor_99_excluidos_analisis.csv")

        # Guardar resumen de validación
        RESUMEN_EDADES.append({
            "archivo": nombre,
            "columna_edad": col_edad,
            "edad_minima_observada_valida": edad_min,
            "edad_minima_referencia": edad_min_ref,
            "edad_maxima": 99,
            "edades_validas": int(mask_valida.sum()),
            "mayores_99": int(mask_mayor.sum()),
            "negativas": int(mask_neg.sum()),
            "no_numericas": int(mask_no_num.sum()),
            "ausentes": int(mask_vacia.sum())
        })

    else:
        print(f"Advertencia: no se encontró columna de edad en {nombre}")
        base["_observacion_edad"] = "No se encontró columna de edad"

    resultados[nombre]["base"] = base
    print(f"Validación terminada: {nombre} | Filas: {len(base)}")


# 22. CREAR RESUMEN GENERAL DE EDADES

df_resumen_edades = pd.DataFrame(RESUMEN_EDADES)

print("\nRESUMEN DE VALIDACIÓN DE EDADES SSC")
display(df_resumen_edades)


# 23. EXPORTAR RESUMEN DE EDADES

if not df_resumen_edades.empty:
    escribir(df_resumen_edades, OUT / "resumen_validacion_edades_SSC.csv")
    print("Resumen de edades guardado correctamente.")
else:
    print("No se encontraron columnas de edad.")


# 24. REGISTRAR VALIDACIÓN EN LA BITÁCORA

for resumen in RESUMEN_EDADES:
    total = len(resultados[resumen["archivo"]]["base"])

    loggear(
        archivo=resumen["archivo"],
        paso="Validación de edades",
        regla="SSC-R04",
        antes=total,
        despues=total,
        decision="Clasificación de registros",
        justificacion="Se identificaron edades válidas, ausentes, negativas, no numéricas y mayores a 99 sin modificar los valores originales.",
        detalle=(
            f"Válidas: {resumen['edades_validas']}; "
            f"Mayores 99: {resumen['mayores_99']}; "
            f"Negativas: {resumen['negativas']}; "
            f"No numéricas: {resumen['no_numericas']}; "
            f"Ausentes: {resumen['ausentes']}"
        )
    )


# 25. ACTUALIZAR BITÁCORA AUTOMÁTICA

df_bitacora = pd.DataFrame(BITACORA)
escribir(df_bitacora, LOG_DIR / "bitacora_SSC.csv")


# 26. MOSTRAR RESULTADO FINAL

print("\n========================================")
print("VALIDACIÓN DE EDADES COMPLETADA")
print("========================================")
print("Archivos procesados:", len(resultados))
print("Archivos con columna de edad:", len(RESUMEN_EDADES))
print("Carpeta de resultados:", OUT)
print("Carpeta de bitácora:", LOG_DIR)
print("Bitácora actualizada correctamente.")


## 4. Aplicación de reglas y clasificación fila por fila

Cada fila no duplicada recibe una decisión primaria: `conservado`, `pendiente` o `excluido`. Se guarda el código de regla y el motivo. Los archivos de pendientes y excluidos son independientes del conjunto limpio. Los faltantes, SD y ceros se distinguen y nunca se imputan automáticamente.

In [ ]:

if RESUMEN_EDADES:
    escribir(pd.DataFrame(RESUMEN_EDADES),OUT/"resumen_validacion_edades.csv")


## 5. Exportación, conciliación y bitácora

La conciliación utiliza categorías mutuamente excluyentes: conservados + pendientes + excluidos + duplicados exactos retirados = registros iniciales. Los folios repetidos son una marca de revisión superpuesta y no se suman como otra categoría.

In [ ]:

# 27. DIAGNÓSTICO DE ESPACIO Y ARCHIVOS TEMPORALES

from pathlib import Path

import os

print("CARPETA DE RESULTADOS:", OUT)
print("\nARCHIVOS CSV GENERADOS:\n")

archivos = sorted(
    OUT.glob("*.csv"),
    key=lambda p: p.stat().st_size,
    reverse=True
)

for archivo in archivos:
    tamano = archivo.stat().st_size / (1024 ** 2)
    print(f"{archivo.name}: {tamano:.2f} MB")

total_mb = sum(
    archivo.stat().st_size for archivo in archivos
) / (1024 ** 2)

print("\nEspacio ocupado por CSV:", round(total_mb, 2), "MB")

# Revisar archivos temporales de prueba
print("\nARCHIVOS DE PRUEBA:")

for archivo in OUT.glob("*prueba*.csv"):
    print(archivo.name, round(
        archivo.stat().st_size / (1024 ** 2), 2
    ), "MB")


In [ ]:
# 28. REVISAR EL ESPACIO DE EJECUCIONES ANTERIORES

from pathlib import Path

carpetas_revision = [
    ROOT / "data" / "processed" / "SSC",
    ROOT / "bitacora" / "SSC"
]

for carpeta in carpetas_revision:
    print("\nCARPETA:", carpeta)

    if not carpeta.exists():
        print("No existe.")
        continue

    total_general = 0

    for subcarpeta in sorted(carpeta.iterdir()):
        if subcarpeta.is_dir():
            peso = sum(
                archivo.stat().st_size
                for archivo in subcarpeta.rglob("*")
                if archivo.is_file()
            )

            total_general += peso

            print(
                subcarpeta.name,
                "->",
                round(peso / (1024 ** 2), 2),
                "MB"
            )

    print(
        "TOTAL:",
        round(total_general / (1024 ** 2), 2),
        "MB"
    )


In [ ]:
# 29. DIAGNÓSTICO DE ALMACENAMIENTO Y ESCRITURA

import shutil
import tempfile
from pathlib import Path

print("DIRECTORIO ACTUAL:")
print(ROOT)

print("\nESPACIO DEL SISTEMA:")

uso = shutil.disk_usage(ROOT)

print("Total:", round(uso.total / (1024**3), 2), "GB")
print("Usado:", round(uso.used / (1024**3), 2), "GB")
print("Libre:", round(uso.free / (1024**3), 2), "GB")

print("\nPRUEBA DE ESCRITURA:")

prueba = ROOT / "prueba_almacenamiento.txt"

try:
    with open(prueba, "w", encoding="utf-8") as archivo:
        archivo.write("Prueba de almacenamiento.")

    print("Escritura correcta.")

    prueba.unlink()
    print("Archivo de prueba eliminado.")

except OSError as error:
    print("Error al escribir:", repr(error))

print("\nCARPETA TEMPORAL:")

try:
    with tempfile.NamedTemporaryFile(
        mode="w",
        encoding="utf-8",
        delete=True
    ) as archivo:
        archivo.write("Prueba temporal.")

    print("La carpeta temporal permite escribir.")

except OSError as error:
    print("Error en carpeta temporal:", repr(error))


## 6. Interpretación y limitaciones

Los conteos definitivos son los que aparecen al ejecutar este notebook con los archivos originales. Los valores reportados en Avance 02 (por ejemplo, 3,498 repeticiones en SSC personas por edad, 145 en ATUS o 76,204 discrepancias C5) son antecedentes que deben volver a comprobarse, no resultados que se copian como si fueran actuales.